# Named Entity Recognition (NER) with Word Embeddings

**Goal:** teach a model to find *named entities* in English text: people (PER), organisations (ORG), locations (LOC) and other names (MISC, e.g. nationalities or events).

For example, in *"Paul Kagame met the president of Kenya in Nairobi."* we want: `Paul Kagame` = PER, `Kenya` = LOC, `Nairobi` = LOC.

**Plan:**
1. Load and explore the CoNLL-2003 dataset
2. Baseline model **without** embeddings (the "before")
3. Model **with GloVe** embeddings (the "after")
4. Model with **FastText** embeddings, and how each handles unknown words
5. Compare all three and look at mistakes
6. Test on a small set of Rwandan sentences
7. Try the best model on new sentences

## Step 1: Load the dataset
We import the libraries we need. `datasets` downloads data from Hugging Face, and `seqeval` scores NER predictions per entity type.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import Counter

import numpy as np
import pandas as pd
import datasets
from datasets import load_dataset

datasets.disable_progress_bars()  # keep the notebook output clean

We load **CoNLL-2003**, the standard English NER dataset (news articles from Reuters).

The original Hugging Face version (`conll2003`) uses an old "loading script" that newer versions of `datasets` refuse to run. So we load Hugging Face's automatic **parquet copy of the same dataset** (`eriktks/conll2003`, revision `refs/convert/parquet`). It has the same sentences, the same labels and the same official train / validation / test splits.

In [2]:
conll = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
conll

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Each sentence is a list of **tokens** (words and punctuation) and a matching list of **NER tags**. The tags are stored as numbers, so we turn them back into their names (like `B-PER`).

In [3]:
TAG_NAMES = conll["train"].features["ner_tags"].feature.names
print("Tag set:", TAG_NAMES)

def to_sentences(split):
    """Return two lists: the tokens of each sentence and their tag names."""
    tokens = [list(t) for t in split["tokens"]]
    tags = [[TAG_NAMES[i] for i in seq] for seq in split["ner_tags"]]
    return tokens, tags

train_tokens, train_tags = to_sentences(conll["train"])
val_tokens, val_tags = to_sentences(conll["validation"])
test_tokens, test_tags = to_sentences(conll["test"])

for word, tag in zip(train_tokens[0], train_tags[0]):
    print(f"{word:10} {tag}")

Tag set: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


EU         B-ORG
rejects    O
German     B-MISC
call       O
to         O
boycott    O
British    B-MISC
lamb       O
.          O


### What do the tags mean? (the BIO format)
- `B-` means the **B**eginning of an entity, and `I-` means **I**nside it (the 2nd, 3rd... word of the same entity).
- `O` means **O**utside: the word is not part of any entity.

So "Paul Kagame" is tagged `B-PER I-PER`. There are 4 entity types: **PER** (person), **ORG** (organisation), **LOC** (location) and **MISC** (other names, like "German" or "World Cup").

**Note:** CoNLL-2003 has **no DATE type**. Words like "Friday" or "1996" are tagged `O`, so none of our models can learn to find dates.

### How big is each split?

In [4]:
def count_entities(tag_seqs):
    """Count entities by counting the B- tags (each entity starts with exactly one)."""
    counts = Counter(t[2:] for seq in tag_seqs for t in seq if t.startswith("B-"))
    return {etype: counts[etype] for etype in ["PER", "ORG", "LOC", "MISC"]}

rows = []
for name, toks, tags in [("train", train_tokens, train_tags),
                         ("validation", val_tokens, val_tags),
                         ("test", test_tokens, test_tags)]:
    row = {"split": name, "sentences": len(toks), "tokens": sum(len(s) for s in toks)}
    row.update(count_entities(tags))
    rows.append(row)

split_summary = pd.DataFrame(rows).set_index("split")
split_summary

,sentences,tokens,PER,ORG,LOC,MISC
split,,,,,,
train,14041,203621,6600,6321,7140,3438
validation,3250,51362,1842,1341,1837,922
test,3453,46435,1617,1661,1668,702


How common is each tag? Most tokens are `O`, so the dataset is very **imbalanced**. That's why we judge models with entity-level precision, recall and F1, not plain accuracy: a model that tags everything `O` would already have high accuracy.

In [5]:
tag_counts = Counter(t for seq in train_tags for t in seq)
tag_share = pd.DataFrame({"count": pd.Series(tag_counts)}).loc[TAG_NAMES]
tag_share["percent"] = (100 * tag_share["count"] / tag_share["count"].sum()).round(2)
tag_share

,count,percent
O,169578,83.28
B-PER,6600,3.24
I-PER,4528,2.22
B-ORG,6321,3.10
I-ORG,3704,1.82
B-LOC,7140,3.51
I-LOC,1157,0.57
B-MISC,3438,1.69
I-MISC,1155,0.57


## Step 2: Baseline model WITHOUT embeddings (the "before")

The model looks at **one word at a time** and guesses its tag. To help, we describe each word with a small dictionary of **features**:

- **Who the word is:** the word in lowercase, the previous word and the next word (so the model sees a bit of context), and the word's last 3 letters (its suffix, e.g. "-ing" or "-ese").
- **Spelling features:** does it start with a capital letter? Is it ALL CAPS? Does it contain a digit or a hyphen? Is it the first word of the sentence? We check the neighbours' capitals too.

Here the model knows words only as **separate labels**: "Kigali" and "Nairobi" are just two different, unrelated features. That's the "no embeddings" setup.

First, the spelling features. We'll **reuse exactly these** in the embedding models later.

In [6]:
def spelling_features(sentence, i):
    """Simple spelling (shape) features for the word at position i and its neighbours."""
    word = sentence[i]
    feats = {
        "is_title": word[:1].isupper(),        # starts with a capital letter
        "is_upper": word.isupper(),            # ALL CAPS, e.g. "WASAC"
        "is_lower": word.islower(),
        "has_digit": any(ch.isdigit() for ch in word),
        "has_hyphen": "-" in word,
        "is_punct": not any(ch.isalnum() for ch in word),
        "is_first": i == 0,                    # capitals at the start of a sentence mean less
        "is_last": i == len(sentence) - 1,
    }
    if i > 0:
        feats["prev_is_title"] = sentence[i - 1][:1].isupper()
        feats["prev_is_upper"] = sentence[i - 1].isupper()
    if i < len(sentence) - 1:
        feats["next_is_title"] = sentence[i + 1][:1].isupper()
        feats["next_is_upper"] = sentence[i + 1].isupper()
    return {name: float(value) for name, value in feats.items()}

def baseline_features(sentence, i):
    """Baseline: who the word is (word, neighbours, suffix) + spelling features."""
    feats = spelling_features(sentence, i)
    feats["word=" + sentence[i].lower()] = 1.0
    feats["suffix3=" + sentence[i][-3:].lower()] = 1.0
    feats["prev=" + (sentence[i - 1].lower() if i > 0 else "<START>")] = 1.0
    feats["next=" + (sentence[i + 1].lower() if i < len(sentence) - 1 else "<END>")] = 1.0
    return feats

print(baseline_features(["Paul", "Kagame", "visited", "Kigali", "."], 3))

{'is_title': 1.0, 'is_upper': 0.0, 'is_lower': 0.0, 'has_digit': 0.0, 'has_hyphen': 0.0, 'is_punct': 0.0, 'is_first': 0.0, 'is_last': 0.0, 'prev_is_title': 0.0, 'prev_is_upper': 0.0, 'next_is_title': 0.0, 'next_is_upper': 0.0, 'word=kigali': 1.0, 'suffix3=ali': 1.0, 'prev=visited': 1.0, 'next=.': 1.0}


Now we turn every word in every sentence into a feature dictionary. `DictVectorizer` then converts these dictionaries into a big table of numbers (one column per possible feature). This is the same "lots of mostly-zero columns" idea as TF-IDF in my spam classifier.

In [7]:
from sklearn.feature_extraction import DictVectorizer

def sentences_to_dicts(sentences, feature_fn):
    return [feature_fn(sent, i) for sent in sentences for i in range(len(sent))]

def flatten(tag_seqs):
    return [t for seq in tag_seqs for t in seq]

vectorizer = DictVectorizer()
X_train_base = vectorizer.fit_transform(sentences_to_dicts(train_tokens, baseline_features))
X_val_base = vectorizer.transform(sentences_to_dicts(val_tokens, baseline_features))
X_test_base = vectorizer.transform(sentences_to_dicts(test_tokens, baseline_features))
y_train = flatten(train_tags)

print("Training table:", X_train_base.shape[0], "words x", X_train_base.shape[1], "features")

Training table: 203621 words x 65784 features


### Training, and choosing a setting on the validation set
We use **Logistic Regression**, a simple and fast linear classifier. It has one main setting, `C`: a small `C` keeps the model simple, and a large `C` lets it fit the training data more closely.

We try a few values of `C` and keep the one with the best F1 on the **validation** set. We don't touch the test set while choosing, so the test score stays a fair "exam on unseen data".

`seqeval` scores at the **entity** level: "Paul Kagame" only counts as correct if *both* words get the right tags.

In [8]:
import time
from sklearn.linear_model import LogisticRegression
from seqeval.metrics import classification_report, f1_score

def regroup(flat_tags, sentences):
    """Cut a flat list of predicted tags back into one list per sentence."""
    out, pos = [], 0
    for sent in sentences:
        out.append(list(flat_tags[pos:pos + len(sent)]))
        pos += len(sent)
    return out

def train_with_validation(X_train, y_train, X_val, val_sentences, val_tags, Cs):
    """Train one model per C, report validation F1, return the best model."""
    best_model, best_f1 = None, -1
    for C in Cs:
        start = time.time()
        model = LogisticRegression(C=C, max_iter=1000)
        model.fit(X_train, y_train)
        val_f1 = f1_score(val_tags, regroup(model.predict(X_val), val_sentences))
        print(f"C={C:<6} validation F1 = {val_f1:.4f}   ({time.time() - start:.0f}s)")
        if val_f1 > best_f1:
            best_model, best_f1 = model, val_f1
    print("Best C:", best_model.C)
    return best_model

baseline_model = train_with_validation(X_train_base, y_train, X_val_base,
                                       val_tokens, val_tags, Cs=[0.1, 1, 10, 100])

C=0.1    validation F1 = 0.7436   (27s)


C=1      validation F1 = 0.8258   (21s)


C=10     validation F1 = 0.8376   (20s)


C=100    validation F1 = 0.8319   (18s)
Best C: 10


### Test-set results for the baseline
Now we run the chosen model **once** on the test set. For each entity type:
- **Precision:** of the entities the model found, how many were right?
- **Recall:** of the real entities, how many did the model find?
- **F1:** a single score that balances the two.

In [9]:
baseline_test_pred = regroup(baseline_model.predict(X_test_base), test_tokens)
print(classification_report(test_tags, baseline_test_pred, digits=3))

results = {}  # we collect every model's test report here for the final comparison
results["Baseline (no embeddings)"] = classification_report(test_tags, baseline_test_pred, output_dict=True)

              precision    recall  f1-score   support

         LOC      0.801     0.815     0.808      1668
        MISC      0.730     0.775     0.752       702
         ORG      0.627     0.650     0.638      1661
         PER      0.711     0.811     0.758      1617

   micro avg      0.715     0.760     0.737      5648
   macro avg      0.717     0.763     0.739      5648
weighted avg      0.715     0.760     0.737      5648

